# Preprocessing + Analysis

This notebook will perform any extra necessary preprocessing steps such as removing null values and stratifying the data with random sampling before performing analysis.

### Pre-processing

In [9]:
import pandas as pd
import os

# Read in the csv file 
base_dir = os.path.dirname(os.path.abspath("__file__")) # Set base directory (required for the Makefile to recognize the path)
file_path = os.path.join(base_dir, "..", "DATA", "uva_reviews_with_sentiment.csv")
df = pd.read_csv(file_path)

# Preview of the data
print(df.to_string)

<bound method DataFrame.to_string of     department       group     course              date  rating  \
0           CS        STEM    CS 2100  Updated 12/20/25    2.33   
1           CS        STEM    CS 2100   Updated 1/08/26    4.33   
2           CS        STEM    CS 2100   Updated 4/11/23    1.00   
3           CS        STEM    CS 2100   Updated 4/12/23    2.33   
4           CS        STEM    CS 2100  Updated 12/16/23    4.00   
..         ...         ...        ...               ...     ...   
514       PSYC  Humanities  PSYC 3420   Updated 5/18/21    2.00   
515       PSYC  Humanities  PSYC 3420  Updated 12/13/21    1.33   
516       PSYC  Humanities  PSYC 3420  Updated 12/21/21    2.33   
517       PSYC  Humanities  PSYC 3420  Updated 12/21/21    3.33   
518       PSYC  Humanities  PSYC 3420   Updated 7/25/24    2.33   

                                                  text  \
0    I took this class in Fall 2025 but for some re...   
1    I took this course F25 with Professor

In [10]:
# Drop any null values 
df = df.dropna(how='any',axis=0) 

We will be using stratified random sampling to reduce bias in the data and ensure that larger groups won't skew the sentiment score. To do this we will first find the department (stratum) with the smallest number of reviews (n)

In [3]:
departments = ['CS', 'BIOL', 'CHEM', 'APMA','PSYC', 'SOC', 'PHIL', 'HIST']
# Iterate through the departments and find n (should be SOC)
n = (
    df[df['department'].isin(departments)] # All data within the departments we are looking for
    .groupby('department') # Aggregates data by department
    .size() # Gets the size or number of reviews per department
    .min() # The smalleset value i.e. smallest number of reviews (n)
)

print(f'n = {n}')
    

n = 34


Now that we have n defined we can randomly sample each of the departments.

In [4]:
# Randomly samples each of the departments for further analyzation
strat_df = (
    df[df['department'].isin(departments)] # All data within the departments we are looking for
    .groupby('department', group_keys=False) # Aggregates data by department
    .sample(n=n, random_state=1) # Random sampling of the data with size n we found earlier
)

### Analysis

Now that we have the data properly cleaned and filtered, we can calculate the average sentiment scores per department.

In [5]:
# Group the data by department and sentiment score then calculate the average values
dept_sentiment = (
    strat_df # The data frame we used stratified random sampling on
    .groupby('department', as_index=False) # Grouping by department
    .agg(avg_sentiment=('sentiment_score', 'mean')) # Aggregates the values and computes a mean score
)

print(dept_sentiment.head)


<bound method NDFrame.head of   department  avg_sentiment
0       APMA       0.533891
1       BIOL       0.626362
2       CHEM       0.501779
3         CS       0.544824
4       HIST       0.781679
5       PHIL       0.771529
6       PSYC       0.592953
7        SOC       0.752350>


Then we split the sentiment scores into the STEM or Humanities side.

In [6]:
# Differentiate data into STEM and Humanities groups
stem = ['CS', 'BIOL', 'CHEM', 'APMA']

# Create a new column called 'group' that specifies whether the department is STEM or Humanities
dept_sentiment['group'] = dept_sentiment['department'].apply(
    lambda x: 'STEM' if x in stem else 'Humanities'
)

print(dept_sentiment)

  department  avg_sentiment       group
0       APMA       0.533891        STEM
1       BIOL       0.626362        STEM
2       CHEM       0.501779        STEM
3         CS       0.544824        STEM
4       HIST       0.781679  Humanities
5       PHIL       0.771529  Humanities
6       PSYC       0.592953  Humanities
7        SOC       0.752350  Humanities


Finally we group them into two lists ready for the t-test.

In [7]:
# Separate mean scores
group_scores = (
    dept_sentiment # Sentiment data
    .groupby('group')['avg_sentiment'] # Groups by the group (STEM vs Humanties) and the Avg Sentiment
    .apply(list) # Turns it into a list
)
# Indexes the lists to separate the scores
stem_scores = group_scores['STEM']
humanities_scores = group_scores['Humanities']

print(f'STEM: {stem_scores} \n Humanities: {humanities_scores}')


STEM: [0.5338911764705883, 0.6263617647058823, 0.5017794117647059, 0.5448235294117647] 
 Humanities: [0.7816794117647059, 0.7715294117647059, 0.5929529411764706, 0.75235]


Here we perform the T-test and output the results into a text file for viewing.

In [8]:
import os
from scipy.stats import ttest_ind

# Perform t-test
t_stat, p_value = ttest_ind(
    stem_scores, # Average STEM scores
    humanities_scores, # Average Humanities scores
    equal_var=False # Does not assume equal variance in the two groups (Welch's)
)

alpha = 0.05 # Alpha value
result_msg = "Null Hypothesis Rejected: There is a statistically significant difference." if p_value < alpha else "Failed to reject the Null Hypothesis: No significant difference detected."

# Calculates an overall average score for the departments
overall_stem_avg = sum(stem_scores) / len(stem_scores) 
overall_hum_avg = sum(humanities_scores) / len(humanities_scores)

# Results for viewing (printing)
results = f"""===============================
STATISTICAL ANALYSIS RESULTS
===============================
T-statistic: {t_stat:.4f}
P-value: {p_value:.4f}
Result: {result_msg}

-------------------------------
AVERAGE SENTIMENT SCORES
-------------------------------
Overall STEM: {overall_stem_avg:.4f}
Overall Humanities: {overall_hum_avg:.4f}
===============================
"""


# Defining paths for saving results
notebook_dir = os.path.abspath('')
output_path = os.path.join(notebook_dir, "..", "OUTPUT", "analysis_results.txt")
# Make sure the path exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)
# Write the file
with open(output_path, "w") as f:
    f.write(results)

print(f"Analysis results successfully saved to: {output_path}")

Analysis results successfully saved to: /Users/jam/Desktop/ds/DS4002Project_1/SCRIPTS/../OUTPUT/analysis_results.txt
